In [4]:
# 1️⃣ Install required packages
!pip install -q langchain==0.3.27 langchain-community langchain-huggingface transformers torch sentence-transformers faiss-cpu

# 2️⃣ Imports
import time
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 3️⃣ Load your document (upload DSML.txt in Colab)

loader = TextLoader("DSML.txt")
docs = loader.load()

# 4️⃣ Split into chunks

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

# 5️⃣ Create embeddings + FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# 6️⃣ Load Phi-2 model

generator = pipeline("text-generation", model="microsoft/phi-2", max_new_tokens=256, do_sample=False)

# 7️⃣ Helper: Extract clean text

def extract_generated_text(raw_output, prompt):
    text = raw_output.replace(prompt, "").strip()
    lines = text.split("\n")
    clean = []
    for line in lines:
        line_strip = line.strip()
        if (
            not line_strip
            or line_strip.lower().startswith("answer:")
            or "you are a helpful assistant" in line_strip.lower()
            or line_strip.startswith("--------------------")
            or line_strip.lower().startswith("question:")
            or line_strip.lower().startswith("solution:")
        ):
            continue
        clean.append(line_strip)
    return " ".join(clean).strip()

# 8️⃣ RAG Response (STRICT MODE)

def rag_response(query, k=3):
    start = time.time()
    retrieved_docs = vectorstore.similarity_search(query, k=k)
    if not retrieved_docs:
        return "I don’t know the answer.", round(time.time() - start, 2)

    context = " ".join([d.page_content for d in retrieved_docs])

    # If similarity is too weak → skip answering
    if len(context.strip()) < 50:
        return "I don’t know the answer.", round(time.time() - start, 2)

    # Check if query terms appear in context — adds stricter filtering
    keywords = [word.lower() for word in query.split() if len(word) > 3]
    match_count = sum(1 for kw in keywords if kw in context.lower())
    if match_count == 0:
        return "I don’t know the answer.", round(time.time() - start, 2)

    # Build prompt
    answer_prompt = f"""
Answer the user's question using ONLY the context below.
If the context does NOT clearly contain the answer, reply exactly: "I don’t know the answer".
Be concise and factual.

Context:
{context}

Question: {query}

Answer:
"""
    output = generator(answer_prompt, max_new_tokens=256, do_sample=False)
    raw_text = output[0]["generated_text"]
    answer = extract_generated_text(raw_text, answer_prompt)

    # If model still generates unrelated text → fallback
    if not answer or "pm of india" in answer.lower() or len(answer.split()) < 3:
        answer = "I don’t know the answer."

    end = time.time()
    return f"{answer}\n_(Response time: {round(end - start, 2)} sec)_"

# 🔟 Chat Loop
print("✅ Strict Phi-2 RAG Chatbot ready! Type 'exit' to quit.\n")

while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("Bot: Session ended. Goodbye!")
        break
    response = rag_response(query)
    print(f"Bot: {response}\n")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


✅ Strict Phi-2 RAG Chatbot ready! Type 'exit' to quit.

You: machine learning


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bot: Machine learning is a subfield of statistical learning that focuses on making predictions using large-scale data.
_(Response time: 1.05 sec)_

You: pm of india
Bot: ('I don’t know the answer.', 0.01)

You: data science


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bot: Possible answer: Data science is a field that involves using algorithms and computer resources to learn from data. It is closely related to statistics, mathematics, and machine learning. Data science projects usually follow these steps: - Gather data to gain information about a research question - Clean, summarize, and visualize the data - Model and analyze the data using various mathematical and statistical methods - Understand the underlying key ideas and algorithms that form the basis of data science Python is a popular programming language for data science because it is easy to use, has many useful packages for data manipulation, and has a gentle introduction for beginners.
_(Response time: 4.26 sec)_

You: EDA?
Bot: ('I don’t know the answer.', 0.01)

You: MONTE CARLO METHODS


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Bot: Monte Carlo methods are a class of computational algorithms that rely on repeated random sampling to obtain numerical results. They are widely used in various fields, including physics, engineering, finance, and computer science. Monte Carlo methods are particularly useful when dealing with complex problems that cannot be easily solved analytically. By simulating a large number of random scenarios, these methods can provide accurate estimates and predictions. One of the key advantages of Monte Carlo methods is their ability to handle uncertainty and randomness. They can be used to model and analyze systems with probabilistic behavior, such as the behavior of financial markets or the spread of diseases. In Monte Carlo simulations, random numbers are generated to represent the uncertain variables in a problem. These random numbers are then used to perform calculations and generate output. The results of the simulations are typically represented as probability distributions, which pr

In [5]:
with open("requirements.txt", "w") as f:
    f.write("""\
langchain==0.3.27
langchain-community
langchain-huggingface
transformers
torch
sentence-transformers
faiss-cpu
accelerate
""")


In [6]:
!cat requirements.txt


langchain==0.3.27
langchain-community
langchain-huggingface
transformers
torch
sentence-transformers
faiss-cpu
accelerate


In [7]:
!zip -r RAG_Based_Chatbot.zip rag_chatbot.ipynb DSML.txt requirements.txt
from google.colab import files
files.download("RAG_Based_Chatbot.zip")



	zip warning: name not matched: rag_chatbot.ipynb
  adding: DSML.txt (deflated 66%)
  adding: requirements.txt (deflated 30%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
!ls


DSML.txt  RAG_Based_Chatbot.zip  requirements.txt  sample_data
